# 8.3 示範回答如何把寫法教進權重？


仍問「2+2=?」，這次準備兩套示範：簡短版回答「4」，生動版回答「4，就像兩雙筷子共有四根」。兩版的事實相同，改的是說法。我們想知道，反覆學其中一種回答後，模型是否在沒有額外提醒時也傾向那種寫法。

**監督式微調（SFT）**用人寫的目標回答教模型：在給定提問後，提高目標文字的機率。真正訓練會更新權重；先整理示範，則是訓練前的資料工作。下面先數每版有哪些位置要學。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import render_chat

question = "2+2=?"
answers = {"簡短": "4", "生動": "4，就像兩雙筷子共有四根。"}
for style, answer in answers.items():
    inputs, labels = render_chat(
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    )
    print(style, "總位置", len(inputs), "學習位置", int((labels != -100).sum()))

`render_chat`把角色與文字整理成模型讀的數字序列。`inputs`含完整前文，`labels`給出下一步目標；`-100`表示這個位置不計猜錯代價。`labels != -100`選出有效位置，`sum()`數它們有幾個。預設以UTF-8位元組作文字單位，簡短版總位置10、學習位置2，生動版46、38；學習位置也含回答結束標記，因此「4」不是只有一個目標。

這個差別會影響比較：兩套都用100筆或更新100步，長版仍可能提供更多要學的文字。若想知道風格訓練的效果，要固定起點與題目，記下有效目標數，再用未教過的題目同時檢查內容和寫法。

一個已保存的小實驗提供了具體反例：同一加法起點分別學短答與固定積木句後，對未教過的`2+2=?`，一支回`3`，另一支回`3，像把兩組積木合在一起再數。`。寫法的差別已出現，數字仍錯。它支持「這種寫法能影響權重」，沒有證明學到通用貼切比喻或新加法。

練習將生動示範改成「4，共有四個」，不改題目和真值。先預測有效學習位置會減少，再執行核對。

<details>
<summary>補充：既有實驗的條件與完整紀錄</summary>

我們已另外從相同的49題加法模型各複製一份，保持問題沒有style前綴，分別用短答或固定積木句再更新450次。在未教過的`2+2=?`上，短答模型實際回`3`，生動模型回`3，像把兩組積木合在一起再數。`；問題相同、寫法確實不同，內容卻同樣錯。七題最後檢查中，兩支各有七題符合自己的固定寫法，數值都零題正確。這個基模本來也沒有答對這七題，不能把結果描述成原先會算、改文風才被破壞。

每支都更新450次、每批16題，累計有效學習位置卻是短答16,591個、生動318,991個。這裡數的是回答內每個token目標，含結束標記，不是完整回答的筆數；同一筆反覆學也會重複計入。長版每筆都有大量固定文字，相同步數不是相同監督量；這次展示了寫法可進入權重，沒有隔離成相同有效目標token數的風格比較。[T.5](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.5)提供重跑入口與各支權重。

</details>
